# DP_LR - Differentially Private Logistic Regression

This notebook:
- Loads preprocessed data from data_preprocessor.ipynb
- Loads baseline results from STD_LR
- Trains DP Logistic Regression with different epsilon values
- Runs 30 iterations per epsilon to capture DP stochasticity
- Reports mean ± std for all metrics
- Saves results to JSON with per-run data

In [1]:
# Import Libraries
import time
import numpy as np
import pickle
import json
import warnings
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from diffprivlib.models import LogisticRegression as DPLogisticRegression
warnings.filterwarnings("ignore")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 1. Load Preprocessed Data

In [2]:
# Load data from data_preprocessor.ipynb
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

# DP norm precondition: clip every row to unit L2 norm (||x_i||_2 <= 1).
X_train = X_train / np.maximum(np.linalg.norm(X_train, axis=1, keepdims=True), 1.0)
X_test = X_test / np.maximum(np.linalg.norm(X_test, axis=1, keepdims=True), 1.0)
data_norm = 1.0

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

Training Set: 455 samples
Testing Set: 114 samples
Number of features: 30


## 2. Load Baseline Results

In [3]:
# Load baseline results from STD_LR
try:
    with open('output/std_lr_report.json', 'r') as f:
        baseline = json.load(f)

    baseline_accuracy = baseline['accuracy']
    print("Baseline results loaded successfully!")
    print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
except FileNotFoundError:
    print("Warning: std_lr_report.json not found. Run STD_LR.ipynb first.")
    baseline_accuracy = None
    baseline_f1 = None

Baseline results loaded successfully!
Baseline Accuracy: 0.9123


## 3. Define Epsilon Values and Run Configuration

Lower epsilon = more privacy, more noise, potentially lower accuracy.
Higher epsilon = less privacy, less noise, potentially higher accuracy.

We run 20 iterations per epsilon to capture the variance introduced by the DP mechanism.

In [4]:
# Epsilon values to test
# epsilon_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
epsilon_values = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 4.0, 6.0, 8.0, 10.0]


N_RUNS = 30  # runs per epsilon to capture DP stochasticity

print(f"Epsilon values: {epsilon_values}")
print(f"Runs per epsilon: {N_RUNS}")

Epsilon values: [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 4.0, 6.0, 8.0, 10.0]
Runs per epsilon: 30


## 4. Detailed Evaluation at ε=1.0 (Multiple Runs)

In [5]:
epsilon = 1

detail_accuracies = []
detail_f1s = []
detail_precisions = []
detail_recalls = []
detail_roc_aucs = []
detail_cms = []

print(f"Training DP Logistic Regression {N_RUNS} times with epsilon={epsilon}...")
print("=" * 80)

detail_extra = ExtraMetrics()
for run in range(N_RUNS):
    seed = run * 10 + 42

    dp_model = DPLogisticRegression(
        epsilon=epsilon,
        max_iter=1000,
        random_state=seed, 
        data_norm=1.0
    )

    _t0 = time.perf_counter()
    dp_model.fit(X_train, y_train)
    train_time = time.perf_counter() - _t0
    y_pred = dp_model.predict(X_test)
    y_pred_proba = dp_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    detail_extra.add(y_test, y_pred, model=dp_model, X=X_test, train_time=train_time)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)

    detail_accuracies.append(acc)
    detail_f1s.append(f1)
    detail_precisions.append(prec)
    detail_recalls.append(rec)
    detail_roc_aucs.append(roc_auc)
    detail_cms.append(cm)

    print(f"  Run {run+1:2d}/{N_RUNS} | Acc: {acc:.4f} | F1: {f1:.4f} | "
            f"Prec: {prec:.4f} | Rec: {rec:.4f} | AUC: {roc_auc:.4f}")

print("=" * 80)
print(f"\nResults over {N_RUNS} runs at epsilon={epsilon}:")
print(f"  Accuracy:  {np.mean(detail_accuracies):.4f} ± {np.std(detail_accuracies):.4f}")
print(f"  F1-Score:  {np.mean(detail_f1s):.4f} ± {np.std(detail_f1s):.4f}")
print(f"  Precision: {np.mean(detail_precisions):.4f} ± {np.std(detail_precisions):.4f}")
print(f"  Recall:    {np.mean(detail_recalls):.4f} ± {np.std(detail_recalls):.4f}")
print(f"  ROC-AUC:   {np.mean(detail_roc_aucs):.4f} ± {np.std(detail_roc_aucs):.4f}")

print("Added metrics over the detail runs:")
print(detail_extra.describe())

Training DP Logistic Regression 30 times with epsilon=1...
  Run  1/30 | Acc: 0.7193 | F1: 0.6667 | Prec: 0.5926 | Rec: 0.7619 | AUC: 0.7490
  Run  2/30 | Acc: 0.6579 | F1: 0.4935 | Prec: 0.5429 | Rec: 0.4524 | AUC: 0.6534
  Run  3/30 | Acc: 0.8772 | F1: 0.8409 | Prec: 0.8043 | Rec: 0.8810 | AUC: 0.9425
  Run  4/30 | Acc: 0.7281 | F1: 0.6173 | Prec: 0.6410 | Rec: 0.5952 | AUC: 0.7186
  Run  5/30 | Acc: 0.5175 | F1: 0.1270 | Prec: 0.1905 | Rec: 0.0952 | AUC: 0.3092
  Run  6/30 | Acc: 0.7807 | F1: 0.6269 | Prec: 0.8400 | Rec: 0.5000 | AUC: 0.8558
  Run  7/30 | Acc: 0.7456 | F1: 0.6742 | Prec: 0.6383 | Rec: 0.7143 | AUC: 0.8148
  Run  8/30 | Acc: 0.4825 | F1: 0.4870 | Prec: 0.3836 | Rec: 0.6667 | AUC: 0.5736
  Run  9/30 | Acc: 0.6930 | F1: 0.6237 | Prec: 0.5686 | Rec: 0.6905 | AUC: 0.7791
  Run 10/30 | Acc: 0.7193 | F1: 0.4483 | Prec: 0.8125 | Rec: 0.3095 | AUC: 0.8353
  Run 11/30 | Acc: 0.4035 | F1: 0.1282 | Prec: 0.1389 | Rec: 0.1190 | AUC: 0.1812
  Run 12/30 | Acc: 0.5263 | F1: 0.4130 

In [6]:
# Average confusion matrix
avg_cm = np.mean(detail_cms, axis=0).astype(int)

TN = avg_cm[0, 0]
FP = avg_cm[0, 1]
FN = avg_cm[1, 0]
TP = avg_cm[1, 1]

custom_cm = np.array([
    [TP, FN],
    [FP, TN]
])

print(f"Average Confusion Matrix ({N_RUNS} runs at ε={epsilon}):")
print(custom_cm)

# Comparison with baseline
if baseline_accuracy is not None:
    mean_acc = np.mean(detail_accuracies)
    mean_f1 = np.mean(detail_f1s)
    accuracy_loss = baseline_accuracy - mean_acc

    print()
    print("=" * 60)
    print(f"COMPARISON: Standard vs DP Logistic Regression ({N_RUNS} runs)")
    print("=" * 60)
    print(f"\nStandard Model Accuracy: {baseline_accuracy:.4f}")
    print(f"DP Model Accuracy:       {mean_acc:.4f} ± {np.std(detail_accuracies):.4f}")
    print(f"Accuracy Loss:           {accuracy_loss:.4f} ({accuracy_loss*100:.2f}%)")
    print(f"DP Model F1-Score:       {mean_f1:.4f} ± {np.std(detail_f1s):.4f}")

Average Confusion Matrix (30 runs at ε=1):
[[23 18]
 [18 53]]

COMPARISON: Standard vs DP Logistic Regression (30 runs)

Standard Model Accuracy: 0.9123
DP Model Accuracy:       0.6789 ± 0.1512
Accuracy Loss:           0.2333 (23.33%)
DP Model F1-Score:       0.5593 ± 0.2064


## 5. Epsilon vs Accuracy Analysis (Multiple Runs per Epsilon)

Testing different privacy budget (epsilon) values to observe the trade-off between privacy and accuracy. Each epsilon is tested 20 times to capture DP variance.

In [7]:
# Store aggregated results
results = {
    'epsilon': [],
    'accuracy_mean': [],
    'accuracy_std': [],
    'f1_score_mean': [],
    'f1_score_std': [],
    'precision_mean': [],
    'precision_std': [],
    'recall_mean': [],
    'recall_std': [],
    'roc_auc_mean': [],
    'roc_auc_std': [],
}

# JSON export
all_results = {
    'model_type': 'Differentially Private Logistic Regression',
    'n_runs_per_epsilon': N_RUNS,
    'epsilon_results': {}
}

print(f"Training DP Logistic Regression with {N_RUNS} runs per epsilon value...")
print("=" * 90)

for eps in epsilon_values:
    run_accuracies = []
    run_f1s = []
    run_precisions = []
    run_recalls = []
    run_roc_aucs = []
    run_cms = []

    extra = ExtraMetrics()
    for run in range(N_RUNS):
        seed = run * 10 + 42

        dp_model = DPLogisticRegression(
            epsilon=eps,
            max_iter=1000,
            random_state=seed, 
            data_norm=1.0
        )

        _t0 = time.perf_counter()
        dp_model.fit(X_train, y_train)
        train_time = time.perf_counter() - _t0
        
        # Export model for MIA (only first run per epsilon)
        if run == 0:
            import sys
            import os
            sys.path.append(os.path.abspath(os.path.join('..', '..')))
            from model_exporter import export_model
            export_model(dp_model, f'output/model/dp_lr_model_eps_{eps}.pkl')
        
        y_pred = dp_model.predict(X_test)
        y_pred_proba = dp_model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        extra.add(y_test, y_pred, model=dp_model, X=X_test, train_time=train_time)
        f1 = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        cm = confusion_matrix(y_test, y_pred)

        run_accuracies.append(acc)
        run_f1s.append(f1)
        run_precisions.append(prec)
        run_recalls.append(rec)
        run_roc_aucs.append(roc_auc)
        run_cms.append(cm.tolist())

    # Compute statistics
    acc_mean, acc_std = np.mean(run_accuracies), np.std(run_accuracies)
    f1_mean, f1_std = np.mean(run_f1s), np.std(run_f1s)
    prec_mean, prec_std = np.mean(run_precisions), np.std(run_precisions)
    rec_mean, rec_std = np.mean(run_recalls), np.std(run_recalls)
    auc_mean, auc_std = np.mean(run_roc_aucs), np.std(run_roc_aucs)

    # Store aggregated results
    results['epsilon'].append(eps)
    # Merge in the added metrics; keys the notebook already tracks are skipped.
    for _key, _value in extra.summary(exclude=('roc_auc',)).items():
        results.setdefault(_key, []).append(_value)
    results['accuracy_mean'].append(acc_mean)
    results['accuracy_std'].append(acc_std)
    results['f1_score_mean'].append(f1_mean)
    results['f1_score_std'].append(f1_std)
    results['precision_mean'].append(prec_mean)
    results['precision_std'].append(prec_std)
    results['recall_mean'].append(rec_mean)
    results['recall_std'].append(rec_std)
    results['roc_auc_mean'].append(auc_mean)
    results['roc_auc_std'].append(auc_std)

    # Store in JSON
    all_results['epsilon_results'][str(eps)] = {
        'epsilon': float(eps),
        'n_runs': N_RUNS,
        'accuracy_mean': acc_mean,
        'accuracy_std': acc_std,
        'f1_mean': f1_mean,
        'f1_std': f1_std,
        'precision_mean': prec_mean,
        'precision_std': prec_std,
        'recall_mean': rec_mean,
        'recall_std': rec_std,
        'roc_auc_mean': auc_mean,
        'roc_auc_std': auc_std,
        'per_run_accuracies': run_accuracies,
        'per_run_f1s': run_f1s,
        'per_run_precisions': run_precisions,
        'per_run_recalls': run_recalls,
        'per_run_roc_aucs': run_roc_aucs,
        'per_run_confusion_matrices': run_cms,
        'parameters': {
            'max_iter': 1000,
            'n_runs': N_RUNS
        }
    }
    all_results['epsilon_results'][str(eps)].update(extra.summary(exclude=('roc_auc',)))
    all_results['epsilon_results'][str(eps)].update(extra.per_run(exclude=('roc_auc',)))

    print(f"ε={eps:5.2f} | Acc: {acc_mean:.4f}±{acc_std:.4f} | "
            f"F1: {f1_mean:.4f}±{f1_std:.4f} | "
            f"Prec: {prec_mean:.4f}±{prec_std:.4f} | "
            f"Rec: {rec_mean:.4f}±{rec_std:.4f} | "
            f"AUC: {auc_mean:.4f}±{auc_std:.4f}")

print("=" * 90)
print("Training complete!")

Training DP Logistic Regression with 30 runs per epsilon value...


Model successfully exported to output/model/dp_lr_model_eps_0.1.pkl
ε= 0.10 | Acc: 0.5567±0.2204 | F1: 0.3959±0.2569 | Prec: 0.4870±0.3231 | Rec: 0.4151±0.2705 | AUC: 0.5159±0.2940
Model successfully exported to output/model/dp_lr_model_eps_0.2.pkl
ε= 0.20 | Acc: 0.5898±0.2040 | F1: 0.4261±0.2547 | Prec: 0.4922±0.3036 | Rec: 0.4222±0.2472 | AUC: 0.5452±0.2837
Model successfully exported to output/model/dp_lr_model_eps_0.4.pkl


ε= 0.40 | Acc: 0.6284±0.1786 | F1: 0.4766±0.2347 | Prec: 0.5349±0.2674 | Rec: 0.4619±0.2320 | AUC: 0.5979±0.2604
Model successfully exported to output/model/dp_lr_model_eps_0.6.pkl
ε= 0.60 | Acc: 0.6620±0.1641 | F1: 0.5321±0.2183 | Prec: 0.5672±0.2488 | Rec: 0.5190±0.2142 | AUC: 0.6444±0.2367
Model successfully exported to output/model/dp_lr_model_eps_0.8.pkl
ε= 0.80 | Acc: 0.6205±0.1700 | F1: 0.4598±0.2426 | Prec: 0.5147±0.2865 | Rec: 0.4746±0.2745 | AUC: 0.5894±0.2603
Model successfully exported to output/model/dp_lr_model_eps_1.0.pkl


ε= 1.00 | Acc: 0.6789±0.1512 | F1: 0.5593±0.2064 | Prec: 0.5881±0.2318 | Rec: 0.5587±0.2187 | AUC: 0.6834±0.2143
Model successfully exported to output/model/dp_lr_model_eps_1.2.pkl
ε= 1.20 | Acc: 0.7249±0.1224 | F1: 0.6250±0.1641 | Prec: 0.6449±0.1859 | Rec: 0.6214±0.1727 | AUC: 0.7443±0.1698
Model successfully exported to output/model/dp_lr_model_eps_1.4.pkl


ε= 1.40 | Acc: 0.7626±0.1010 | F1: 0.6712±0.1402 | Prec: 0.6975±0.1606 | Rec: 0.6579±0.1472 | AUC: 0.7900±0.1363
Model successfully exported to output/model/dp_lr_model_eps_1.6.pkl
ε= 1.60 | Acc: 0.7898±0.0841 | F1: 0.7050±0.1192 | Prec: 0.7396±0.1401 | Rec: 0.6825±0.1266 | AUC: 0.8238±0.1121
Model successfully exported to output/model/dp_lr_model_eps_1.8.pkl
ε= 1.80 | Acc: 0.8096±0.0771 | F1: 0.7291±0.1130 | Prec: 0.7717±0.1286 | Rec: 0.6984±0.1218 | AUC: 0.8506±0.0941
Model successfully exported to output/model/dp_lr_model_eps_2.0.pkl


ε= 2.00 | Acc: 0.8278±0.0727 | F1: 0.7523±0.1066 | Prec: 0.8065±0.1258 | Rec: 0.7111±0.1106 | AUC: 0.8726±0.0795
Model successfully exported to output/model/dp_lr_model_eps_4.0.pkl
ε= 4.00 | Acc: 0.8965±0.0225 | F1: 0.8425±0.0374 | Prec: 0.9565±0.0447 | Rec: 0.7556±0.0553 | AUC: 0.9636±0.0232
Model successfully exported to output/model/dp_lr_model_eps_6.0.pkl
ε= 6.00 | Acc: 0.9094±0.0135 | F1: 0.8611±0.0241 | Prec: 0.9858±0.0226 | Rec: 0.7659±0.0413 | AUC: 0.9844±0.0125
Model successfully exported to output/model/dp_lr_model_eps_8.0.pkl


ε= 8.00 | Acc: 0.9114±0.0121 | F1: 0.8638±0.0216 | Prec: 0.9923±0.0145 | Rec: 0.7659±0.0369 | AUC: 0.9910±0.0073
Model successfully exported to output/model/dp_lr_model_eps_10.0.pkl
ε=10.00 | Acc: 0.9149±0.0109 | F1: 0.8696±0.0187 | Prec: 0.9961±0.0100 | Rec: 0.7722±0.0305 | AUC: 0.9941±0.0043
Training complete!


## 6. Summary of Results

In [8]:
import pandas as pd

results_df = pd.DataFrame(results)

if baseline_accuracy is not None:
    results_df['accuracy_loss_mean'] = baseline_accuracy - results_df['accuracy_mean']
    results_df['accuracy_loss_pct'] = results_df['accuracy_loss_mean'] * 100

print(f"\nResults Summary ({N_RUNS} runs per epsilon):")
print(results_df.to_string(index=False))


Results Summary (30 runs per epsilon):
 epsilon  accuracy_mean  accuracy_std  f1_score_mean  f1_score_std  precision_mean  precision_std  recall_mean  recall_std  roc_auc_mean  roc_auc_std  balanced_accuracy_mean  balanced_accuracy_std  train_time_mean  train_time_std  accuracy_loss_mean  accuracy_loss_pct
     0.1       0.556725      0.220369       0.395870      0.256896        0.486982       0.323082     0.415079    0.270472      0.515873     0.294041                0.527216               0.200829         0.000727        0.000119            0.355556          35.555556
     0.2       0.589766      0.203950       0.426095      0.254714        0.492162       0.303627     0.422222    0.247202      0.545161     0.283670                0.554861               0.197253         0.000669        0.000076            0.322515          32.251462
     0.4       0.628363      0.178619       0.476602      0.234653        0.534895       0.267381     0.461905    0.231953      0.597950     0.260445    

## 7. Save Results to JSON

In [9]:
import os
os.makedirs('output', exist_ok=True)
# Save to JSON
with open('output/dp_lr_report.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print("Results saved to output/dp_lr_report.json")
print(f"Total epsilon values: {len(all_results['epsilon_results'])}")
print(f"Runs per epsilon: {N_RUNS}")

# Also save CSV for easy analysis
results_df.to_csv('output/dp_results.csv', index=False)
print("CSV saved to output/dp_results.csv")

Results saved to ./data/dp_lr_report.json
Total epsilon values: 15
Runs per epsilon: 30
CSV saved to ./data/dp_lr_results.csv


In [10]:
# Verify saved results
with open('output/dp_lr_report.json', 'r') as f:
    loaded_results = json.load(f)

print("Saved Results Verification:")
print(f"  Model Type: {loaded_results['model_type']}")
print(f"  Runs per epsilon: {loaded_results['n_runs_per_epsilon']}")
print(f"  Epsilon values tested: {list(loaded_results['epsilon_results'].keys())}")

print(f"\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")
if baseline_accuracy is not None:
    print(f"\nBaseline (Standard LR) Accuracy: {baseline_accuracy:.4f}")
print(f"\nDP Logistic Regression Results ({N_RUNS} runs per epsilon):")
print(results_df.to_string(index=False))

Saved Results Verification:
  Model Type: Differentially Private Logistic Regression
  Runs per epsilon: 30
  Epsilon values tested: ['0.1', '0.2', '0.4', '0.6', '0.8', '1.0', '1.2', '1.4', '1.6', '1.8', '2.0', '4.0', '6.0', '8.0', '10.0']

FINAL SUMMARY

Baseline (Standard LR) Accuracy: 0.9123

DP Logistic Regression Results (30 runs per epsilon):
 epsilon  accuracy_mean  accuracy_std  f1_score_mean  f1_score_std  precision_mean  precision_std  recall_mean  recall_std  roc_auc_mean  roc_auc_std  balanced_accuracy_mean  balanced_accuracy_std  train_time_mean  train_time_std  accuracy_loss_mean  accuracy_loss_pct
     0.1       0.556725      0.220369       0.395870      0.256896        0.486982       0.323082     0.415079    0.270472      0.515873     0.294041                0.527216               0.200829         0.000727        0.000119            0.355556          35.555556
     0.2       0.589766      0.203950       0.426095      0.254714        0.492162       0.303627     0.422222 